# Notebook 10 — Integrated Computational Toxicology Pipeline
**Author: Himanshu Goel** | [Website](https://hgoelgithub.github.io)

This capstone notebook assembles a **production-ready integrated toxicology screening pipeline** mirroring workflows at Pfizer, Novartis, AstraZeneca safety informatics groups and regulatory agencies (FDA NCTR AI4TOX, EPA CompTox).

### Pipeline architecture
```
SMILES input
  [1] Structural alert screening (ICH M7 / MACCS SMARTS)
  [2] ADMET profiling (Ro5, BBB, CNS MPO, metabolic flags)
  [3] Multi-endpoint ML scoring (6 organ toxicity endpoints)
  [4] Applicability domain check (Tanimoto)
  [5] Risk aggregation (Low / Medium / High)
  [6] Radar visualization + HTML report
```

### Regulatory frameworks covered
| Endpoint | Framework | Accepted tool type |
|---|---|---|
| DILI | ICH S2, FDA LTKB | QSAR, structural alert |
| Cardiotox | ICH E14/S7B, CiPA | Multi-channel QSAR |
| Genotox | ICH M7(R2) | Two complementary QSAR |
| Nephrotox | ICH S7A | QSAR + biomarker |
| Neurotox | ICH S7A, OECD 424 | QSAR + MEA |
| Acute tox | GHS, OECD TG | QSAR (3Rs) |

In [ ]:
!pip install rdkit scikit-learn xgboost pandas numpy matplotlib -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, MACCSkeys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

# ── MODULE 1: Structural alert library ─────────────────────────────────────
ALERTS={
    "Genotox|Nitroaromatic":    "[c][$([NX3](=O)=O)]",
    "Genotox|Aromatic amine":   "Nc1ccccc1",
    "Genotox|Hydrazine":        "[NX3][NX3]",
    "Genotox|Epoxide":          "[OX2r3]",
    "Genotox|Alkyl halide":     "[CX4][Cl,Br,I]",
    "Genotox|N-nitroso":        "[NX3][NX2]=O",
    "Hepato|Michael acceptor":  "[CX3](=O)[CX3]=[CX3]",
    "Hepato|Quinone":           "O=C1C=CC(=O)C=C1",
    "Cardio|Basic N-piperidine":"N1CCCCC1",
    "Nephro|Sulfonamide":       "[SX4](=O)(=O)[NX3]",
    "Nephro|Heavy metal":       "[Hg,Pt,Cd,Pb,As]",
    "Neuro|Organophosphate":    "P(=O)([OX2])[OX2]",
    "PAINS|Catechol":           "Oc1ccccc1O",
}

def flag_alerts(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return {}
    hits={}
    for aname,smarts in ALERTS.items():
        try:
            p=Chem.MolFromSmarts(smarts)
            if p and mol.HasSubstructMatch(p):
                cat=aname.split("|")[0]
                hits.setdefault(cat,[]).append(aname.split("|")[1])
        except: pass
    return hits

# ── MODULE 2: ADMET profiling ───────────────────────────────────────────────
def admet(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return None
    mw=Descriptors.ExactMolWt(mol); logp=Descriptors.MolLogP(mol)
    tpsa=Descriptors.TPSA(mol); hbd=rdMolDescriptors.CalcNumHBD(mol)
    hba=rdMolDescriptors.CalcNumHBA(mol); ar=rdMolDescriptors.CalcNumAromaticRings(mol)
    csp3=Descriptors.FractionCSP3(mol)
    ro5=mw<=500 and logp<=5 and hbd<=5 and hba<=10
    bbb=mw<450 and 0<logp<5 and tpsa<90 and hbd<=3
    # CNS MPO (Pfizer, Wager 2010) — 6 desirability functions 0-6
    d=sum([1.0 if mw<=360 else max(0,1-(mw-360)/100),
           1.0 if logp<=3 else max(0,1-(logp-3)/2),
           1.0 if 40<=tpsa<=90 else (tpsa/40 if tpsa<40 else max(0,1-(tpsa-90)/30)),
           1.0 if hbd==0 else max(0,1-hbd/3),
           1.0 if ar<=1 else max(0,1-(ar-1)/2),
           0.5])  # pKa simplified
    return {"MW":round(mw,1),"LogP":round(logp,2),"TPSA":round(tpsa,1),
            "HBD":hbd,"HBA":hba,"Ro5":"PASS" if ro5 else "FAIL",
            "BBB":"Yes" if bbb else "No","CNS_MPO":round(d,1)}

print("Modules 1 and 2 loaded:")
print(f"  {len(ALERTS)} structural alerts across {len(set(k.split('|')[0] for k in ALERTS))} categories")
print("  ADMET profiler: Ro5, BBB, CNS MPO")

In [ ]:
# ── MODULE 3: Pre-trained ML per endpoint ─────────────────────────────────
def featurize(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return None
    fp=np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,2048))
    maccs=np.array(MACCSkeys.GenMACCSKeys(mol))
    pc=np.array([
        Descriptors.ExactMolWt(mol),Descriptors.MolLogP(mol),Descriptors.TPSA(mol),
        rdMolDescriptors.CalcNumHBD(mol),rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),Descriptors.FractionCSP3(mol),
        Descriptors.MolMR(mol),rdMolDescriptors.CalcNumRings(mol),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==16),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in [9,17,35,53]),
    ])
    return np.concatenate([fp,maccs,pc])

# Training data per endpoint (from previous notebooks)
ep_train={
    "DILI":[("CC(=O)Nc1ccc(O)cc1",1),("c1ccc2c(c1)ccc1cccc3cccc2c13",1),
            ("Nc1ccc([N+](=O)[O-])cc1",1),("Nc1ccccc1",1),("NN",1),
            ("CN(C)C(=N)NC(=N)N",0),("Cn1cnc2c1c(=O)n(C)c(=O)n2C",0),
            ("OCC(O)CO",0),("OC(=O)c1ccccc1",0),("CC(=O)OCC",0)],
    "hERG":[("OC(c1ccc(C(c2ccccc2)(c2ccccc2)O)cc1)CCCN1CCC(CC1)C(O)(c1ccccc1)c1ccccc1",1),
            ("CN(CCOc1ccc(NS(=O)(=O)c2ccc(NC)cc2)cc1)S(=O)(=O)c1ccc(N)cc1",1),
            ("CC(=O)Oc1ccccc1C(=O)O",0),("CN(C)C(=N)NC(=N)N",0),
            ("Cn1cnc2c1c(=O)n(C)c(=O)n2C",0),("OCC(O)CO",0)],
    "Genotox":[("[O-][N+](=O)c1ccccc1",1),("c1ccc2c(c1)ccc1cccc3cccc2c13",1),
               ("BrCCBr",1),("NN",1),("Nc1ccccc1",1),
               ("OC(=O)c1ccccc1",0),("CN(C)C(=N)NC(=N)N",0),
               ("OCC(O)CO",0),("CC(=O)OCC",0),("CC(C)=O",0)],
    "Nephrotox":[[("[Pt](Cl)(Cl)(N)N",1),("ClC(Cl)(Cl)Cl",1),
                 ("Cc1ccc(S(=O)(=O)Nc2ccccn2)cc1",1),("CC(=O)Nc1ccc(O)cc1",1),
                 ("OCC(O)CO",0),("CN(C)C(=N)NC(=N)N",0),
                 ("OC(=O)c1ccccc1",0),("CC(=O)OCC",0)]],
    "Neurotox":[("CCOP(=S)(OCC)Oc1nc(Cl)c(Cl)cc1Cl",1),
                ("CNC(=O)Oc1ccc2[nH]c(C)c(C)c2c1",1),
                ("Cn1cnc2c1c(=O)n(C)c(=O)n2C",0),("CC(=O)Oc1ccccc1C(=O)O",0),
                ("OCC(O)CO",0),("CN(C)C(=N)NC(=N)N",0)],
}
# Fix nested list
if isinstance(ep_train["Nephrotox"][0],list): ep_train["Nephrotox"]=ep_train["Nephrotox"][0]

models={}; scalers={}
for ep,data in ep_train.items():
    vd=[(s,l) for s,l in data if featurize(s) is not None]
    if len(vd)<4: continue
    Xe=np.array([featurize(s) for s,_ in vd])
    ye=np.array([l for _,l in vd])
    sc=StandardScaler(); Xes=sc.fit_transform(Xe)
    rf=RandomForestClassifier(200,class_weight='balanced',random_state=42).fit(Xes,ye)
    models[ep]=rf; scalers[ep]=sc

print(f"Module 3: Trained {len(models)} endpoint models: {list(models.keys())}")

In [ ]:
# ── FULL PIPELINE ──────────────────────────────────────────────────────────
train_fps={}
for ep,data in ep_train.items():
    fps=[]
    for s,_ in data:
        mol=Chem.MolFromSmiles(s)
        if mol: fps.append(np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,1024)))
    train_fps[ep]=fps

def tanimoto(fp1,fp2):
    i=(fp1&fp2).sum(); u=(fp1|fp2).sum()
    return i/u if u>0 else 0

def screen(smi, name="?"):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return {"name":name,"error":"Invalid SMILES"}
    result={"name":name,"smiles":smi}

    # 1. Structural alerts
    hits=flag_alerts(smi)
    result["alerts"]=hits; result["n_alerts"]=sum(len(v) for v in hits.values())

    # 2. ADMET
    result["admet"]=admet(smi)

    # 3. ML scores
    feat=featurize(smi)
    scores={}
    for ep,model in models.items():
        fs=scalers[ep].transform([feat])
        scores[ep]=round(model.predict_proba(fs)[0,1],3)
    result["tox_scores"]=scores

    # 4. AD per endpoint
    mol_fp=np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,1024))
    ad_flags={}
    for ep,fps in train_fps.items():
        max_sim=max(tanimoto(mol_fp,fp) for fp in fps) if fps else 0
        ad_flags[ep]="IN" if max_sim>=0.4 else "OOD"
    result["ad"]=ad_flags

    # 5. Risk
    max_sc=max(scores.values()) if scores else 0
    n_high=sum(1 for s in scores.values() if s>0.7)
    if n_high>=2 or max_sc>0.8 or result["n_alerts"]>=3: risk="HIGH"
    elif n_high>=1 or max_sc>0.5 or result["n_alerts"]>=1: risk="MEDIUM"
    else: risk="LOW"
    result["risk"]=risk
    return result

# Screen compound library
library=[
    ("CC(=O)Nc1ccc(O)cc1","Acetaminophen"),
    ("CN(C)C(=N)NC(=N)N","Metformin"),
    ("[O-][N+](=O)c1ccccc1","Nitrobenzene"),
    ("OC(c1ccc(C(c2ccccc2)(c2ccccc2)O)cc1)CCCN1CCC(CC1)C(O)(c1ccccc1)c1ccccc1","Terfenadine"),
    ("Cn1cnc2c1c(=O)n(C)c(=O)n2C","Caffeine"),
    ("CCOP(=S)(OCC)Oc1nc(Cl)c(Cl)cc1Cl","Chlorpyrifos"),
    ("OCC(O)CO","Glycerol"),
    ("Nc1ccccc1","Aniline"),
    ("CC(C)Cc1ccc(cc1)C(C)C(=O)O","Ibuprofen"),
    ("c1ccc2c(c1)ccc1cccc3cccc2c13","Benzo[a]pyrene"),
]
results=[screen(s,n) for s,n in library]

print("INTEGRATED TOXICOLOGY SCREENING RESULTS")
print("="*90)
h="Compound"; ep_headers=" ".join(f"{e[:6]:>7}" for e in models.keys())
print(f"{'Compound':25s} {'Risk':>8} {ep_headers} {'Alerts':>7} {'Ro5':>5}")
print("-"*90)
for r in results:
    if "error" in r: continue
    sc=r["tox_scores"]; ep_cols=" ".join(f"{sc.get(e,0):>7.3f}" for e in models.keys())
    sym="[H]" if r["risk"]=="HIGH" else "[M]" if r["risk"]=="MEDIUM" else "[L]"
    ro5=r.get("admet",{}).get("Ro5","?")
    print(f"{r['name']:25s} {sym}{r['risk']:>6} {ep_cols} {r['n_alerts']:>7} {ro5:>5}")

## Radar chart visualization

In [ ]:
ep_list=list(models.keys())
n=len(ep_list); angles=[i*2*3.14159/n for i in range(n)]+[0]
rc={"HIGH":"#e74c3c","MEDIUM":"#f39c12","LOW":"#27ae60"}

ncols=5; nrows=(len(results)+ncols-1)//ncols
fig=plt.figure(figsize=(4*ncols,4*nrows))
for idx,r in enumerate(results):
    if "error" in r: continue
    ax=fig.add_subplot(nrows,ncols,idx+1,polar=True)
    sc=r["tox_scores"]
    vals=[sc.get(e,0) for e in ep_list]+[sc.get(ep_list[0],0)]
    col=rc[r["risk"]]
    ax.plot(angles,vals,color=col,lw=2)
    ax.fill(angles,vals,color=col,alpha=0.2)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(ep_list,size=7)
    ax.set_ylim(0,1); ax.axhline(0.5,color='gray',lw=0.5,linestyle='--',alpha=0.5)
    ax.set_title(f"{r['name']}\n[{r['risk']}]",size=8,color=col,pad=10)

plt.suptitle("Integrated Toxicology Radar — Library Screen",fontsize=13,y=1.01)
plt.tight_layout(); plt.savefig("tox_radar.png",dpi=150,bbox_inches="tight"); plt.show()

## Generate HTML toxicology report

In [ ]:
# Build HTML report (plain string — no CSS braces issue)
rows=""
for r in results:
    if "error" in r: continue
    sc=r["tox_scores"]
    alert_str="; ".join(f"{k}:{','.join(v)}" for k,v in r.get("alerts",{}).items()) or "None"
    admet_r=r.get("admet",{})
    bg={"HIGH":"#fde8e8","MEDIUM":"#fef9e7","LOW":"#eafaf1"}[r["risk"]]
    border={"HIGH":"#e74c3c","MEDIUM":"#f39c12","LOW":"#27ae60"}[r["risk"]]
    rows+=f'<tr style="background:{bg};border-left:4px solid {border}">'
    rows+=f'<td><b>{r["name"]}</b></td><td><b>{r["risk"]}</b></td>'
    for ep in list(models.keys()):
        v=sc.get(ep,0); cell_bg="#fcc" if v>0.6 else "#ffe" if v>0.4 else "#cfc"
        rows+=f'<td style="background:{cell_bg}">{v:.3f}</td>'
    rows+=f'<td style="font-size:10px">{alert_str[:50]}</td>'
    rows+=f'<td>{admet_r.get("Ro5","?")}</td><td>{admet_r.get("LogP","?")}</td></tr>\n'

ep_th="".join(f"<th>{e}</th>" for e in models.keys())
html_parts=[
    "<!DOCTYPE html><html><head><meta charset='utf-8'>",
    "<title>Computational Toxicology Report</title>",
    "<style>body{font-family:Arial,sans-serif;margin:2rem;color:#1a1a2e}",
    "h1{color:#1565c0}table{width:100%;border-collapse:collapse;font-size:12px}",
    "th{background:#0d2137;color:white;padding:8px;text-align:left}",
    "td{padding:6px 8px;border-bottom:1px solid #e0e0e0}</style></head><body>",
    "<h1>Integrated Computational Toxicology Report</h1>",
    "<p><b>Author:</b> Himanshu Goel | hgoelgithub.github.io | 2025</p>",
    "<p><b>Pipeline:</b> Structural alerts + RF models (ECFP4+MACCS+PC) + AD + risk scoring</p>",
    "<table><tr><th>Compound</th><th>Risk</th>"+ep_th+"<th>Alerts</th><th>Ro5</th><th>LogP</th></tr>",
    rows,
    "</table><p style='font-size:11px;color:#666'>",
    "Disclaimer: Research only. Confirm with experiment. Models outside AD flagged.</p>",
    "</body></html>"
]
html="".join(html_parts)
with open("toxicology_report.html","w") as f: f.write(html)

high=sum(1 for r in results if r.get("risk")=="HIGH")
med=sum(1 for r in results if r.get("risk")=="MEDIUM")
low=sum(1 for r in results if r.get("risk")=="LOW")
print(f"HTML report saved: toxicology_report.html")
print(f"Summary: HIGH={high}  MEDIUM={med}  LOW={low}")

## Industry standards reference table

In [ ]:
summary={
    "Tox21 (NB01)":     "NIH/EPA/FDA benchmark | 12 NR+SR endpoints | DeepTox DNN | ICH S2/S7",
    "DILI (NB02)":      "FDA DILIrank gold standard | ECFP+MACCS+PC | SHAP | ICH S2",
    "Cardiotox (NB03)": "CiPA multi-channel (hERG+Nav+CaL) | IC50 regression | MC Dropout | ICH E14/S7B",
    "Neurotox/BBB (NB04)":"B3DB BBB + AChE | BBB-score | CNS MPO | ICH S7A | OECD 424",
    "Nephrotox (NB05)": "NephroToxDB + HJF guinea pig | KIM-1/NGAL biomarkers | ICH S7A",
    "Genotox (NB06)":   "Hansen Ames 6512 cpds | ICH M7(R2) alerts | Consensus | OECD 471",
    "LD50 (NB07)":      "Zhu 2009 7413 cpds | GHS classification | OECD TG 423 | 3Rs",
    "Multi-task (NB08)":"Shared encoder + task heads | Masked loss | Production DNN | Industry standard",
    "XAI (NB09)":       "SHAP + atom highlight + AD | OECD 5 principles | ICH M7 regulatory",
    "Pipeline (NB10)":  "Alerts+ADMET+ML+AD+Risk scoring | HTML report | IATA/NGRA aligned",
}
print("Computational Toxicology Series — Industry Standards Coverage")
print("="*75)
for nb,desc in summary.items():
    print(f"  {nb:20s} {desc}")

## Key takeaways
- This pipeline covers all major organ toxicity endpoints required for IND filing
- Structural alerts + ML + AD = three complementary evidence streams (weight-of-evidence)
- HTML report generation automates the regulatory documentation workflow
- Next step: integrate with cheminformatics pipeline (Notebooks 1-15 from main series)
- Advanced extensions: PBPK modeling, physiologically-based toxicokinetics (PBTK), AOP networks
- IATA (Integrated Approaches to Testing and Assessment) framework: combine in silico + in vitro + in vivo data